In [1]:
from facebook_business.api import FacebookAdsApi
from facebook_business.adobjects.adaccount import AdAccount
from facebook_business.adobjects.campaign import Campaign
from dotenv import load_dotenv
import os
load_dotenv()

# Step 1: Set up your API credentials
ACCESS_TOKEN = os.getenv('ACCESS_TOKEN')
APP_ID = os.getenv('APP_ID')
APP_SECRET = os.getenv('SECRET_TOKEN')
AD_ACCOUNT_ID = 'act_2353530418276809'  
# AD_ACCOUNT_ID = 'act_234166493'  

# Step 2: Initialize the API
FacebookAdsApi.init(APP_ID, APP_SECRET, ACCESS_TOKEN)

In [5]:
# Step 3: Fetch campaigns
def get_campaigns():
    try:
        campaigns = AdAccount(AD_ACCOUNT_ID).get_campaigns(fields=[
            Campaign.Field.id,
            Campaign.Field.name,
        ])
        for campaign in campaigns:
            print(f"Campaign ID: {campaign[Campaign.Field.id]}, Name: {campaign[Campaign.Field.name]}")
    except Exception as e:
        print(f"Error fetching campaigns: {e}")

# Run the function
get_campaigns()

Campaign ID: 120213265876940368, Name: Talkative Campaign
Campaign ID: 120212031328970368, Name: Instagram post: Prices lower than any e-commerce...


In [2]:
fields = ['name', 'start_time', 'status']

ad_account = AdAccount(AD_ACCOUNT_ID)
# Fetch campaigns and iterate through them
campaigns = ad_account.get_campaigns(fields=fields)

# Print details for each campaign
for campaign in campaigns:
    campaign_id = campaign['id']
    campaign_name = campaign['name']
    start_time = campaign.get('start_time', 'No start time available')
    end_time = campaign.get('end_time', 'No end time available')
    status = campaign.get('status', 'No status available')

    # Output the details
    print(f"Campaign ID: {campaign_id}")
    print(f"Campaign Name: {campaign_name}")
    print(f"Start Time: {start_time}")
    print(f"End Time: {end_time}")
    print(f"Status: {status}")
    print("-" * 50)

Campaign ID: 120213265876940368
Campaign Name: Talkative Campaign
Start Time: 2024-12-04T17:00:00+0530
End Time: No end time available
Status: ACTIVE
--------------------------------------------------
Campaign ID: 120212031328970368
Campaign Name: Instagram post: Prices lower than any e-commerce...
Start Time: 2024-10-19T14:43:58+0530
End Time: No end time available
Status: ACTIVE
--------------------------------------------------


In [ ]:
# Specify the Campaign ID
# campaign_id = '120212031328970368'  # old
campaign_id = '120213265876940368'  # new

# Fields and parameters for insights
fields = [
    'campaign_name',
    'impressions',
    'clicks',
    'spend',
    'reach',
    'ctr',
]
params = {
    'date_preset': 'last_month',  # Adjust date range as needed
    'level': 'campaign',
}

# Fetch Campaign Details
# campaign = Campaign(campaign_id).api_get(fields=['name', 'status'])
# print("Campaign Details:", campaign)

# Fetch Campaign Insights
try:
    insights = Campaign(campaign_id).get_insights(fields=fields, params=params)
    print("\nCampaign Insights:")
    if insights:
        for insight in insights:
            print(insight)
    else:
        print("No insights available for this campaign in the specified time range.")
except Exception as e:
    print("Error fetching insights:", e)


Campaign Insights:
No insights available for this campaign in the specified time range.


## Get Accounts

In [8]:
from facebook_business.adobjects.user import User

me = User(fbid="me")
my_accounts = list(me.get_ad_accounts(
    fields=["account_id", "id", "name"])
)

print(my_accounts)

[<AdAccount> {
    "account_id": "2353530418276809",
    "id": "act_2353530418276809",
    "name": "egenie haripriya"
}]


## Get Campaigns

In [9]:
campaign_list = []
for account in my_accounts:
    my_account = AdAccount(account["id"])
    campaigns = my_account.get_campaigns(
        fields=["id", "account_id", "name", "status"]
    )
    campaign_list.extend(campaigns)

for campaign in campaign_list:
    print(campaign)
    print("-" * 50)

<Campaign> {
    "account_id": "2353530418276809",
    "id": "120213265876940368",
    "name": "Talkative Campaign",
    "status": "ACTIVE"
}
--------------------------------------------------
<Campaign> {
    "account_id": "2353530418276809",
    "id": "120212031328970368",
    "name": "Instagram post: Prices lower than any e-commerce...",
    "status": "ACTIVE"
}
--------------------------------------------------


## Getting Campaigns Insight

In [10]:
import json

# Assuming `insight_list` is a list of AdsInsights objects
def print_insights(insights):
    for idx, insight in enumerate(insights, start=1):
        try:
            # Convert AdsInsights object to dictionary
            insight_data = insight.export_all_data()
            
            # Print insight in a readable JSON format
            print(f"Insight {idx}:")
            print(json.dumps(insight_data, indent=4))
            print("-" * 50)
        except AttributeError:
            print(f"Insight {idx} could not be processed. Invalid format.")
        except Exception as e:
            print(f"Unexpected error processing Insight {idx}: {e}")

In [ ]:
insight_list = []
# campaign_id = '120213265876940368'
campaign_id = '120212031328970368'
my_account = AdAccount(AD_ACCOUNT_ID)
insights_age = my_account.get_insights(
    fields=[
        "campaign_name", "impressions", "clicks", "ctr", "cpc", "reach", 
        "frequency", "spend", "website_ctr", "video_avg_time_watched_actions", 
        "canvas_avg_view_time", "cost_per_unique_inline_link_click", 
        "cost_per_unique_outbound_click", "date_start", "date_stop", 
        "auction_competitiveness", "action_values"
        ],
    params={
            "date_preset": "this_quarter",
            "breakdowns": "age",  
            "level": "campaign",
            'filtering': [{'field': 'campaign.id', 'operator': 'EQUAL', 'value': campaign_id}],
    }
)

d:\ML\Egenie\egenie-chatbot\chatbotvenv\Lib\site-packages\facebook_business\utils\api_utils.py:16: UserWarning: value of breakdowns might not be compatible.  Expect list<breakdowns_enum>;  got <class 'str'>
  warnings.warn(message)


In [22]:
for insight in insights_age:
        #... other code ...
    print(f"Insight:")
    print(insight)
    print("--------------------------------------------------")   
    website_ctr_list = insight.get('website_ctr', [])
    for website_ctr_item in website_ctr_list:
        link_click_value = website_ctr_item.get('value')
        if link_click_value is not None:  
            link_click_value = float(link_click_value) # Convert to float for calculations or comparisons
            print(f"Website CTR (link click) for age {insight.get('age', 'N/A')}: {link_click_value:.2f}%")

Insight:
<AdsInsights> {
    "age": "18-24",
    "campaign_name": "Instagram post: Prices lower than any e-commerce...",
    "clicks": "1132",
    "cost_per_unique_inline_link_click": "9.490041",
    "cpc": "2.624013",
    "ctr": "1.628754",
    "date_start": "2024-10-01",
    "date_stop": "2024-12-04",
    "frequency": "1.531804",
    "impressions": "69501",
    "reach": "45372",
    "spend": "2970.3829",
    "video_avg_time_watched_actions": [
        {
            "action_type": "video_view",
            "value": "5"
        }
    ],
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.46618"
        }
    ]
}
--------------------------------------------------
Website CTR (link click) for age 18-24: 0.47%
Insight:
<AdsInsights> {
    "age": "25-34",
    "campaign_name": "Instagram post: Prices lower than any e-commerce...",
    "clicks": "643",
    "cost_per_unique_inline_link_click": "9.44959",
    "cpc": "2.380768",
    "ctr": "3.602846",

In [ ]:
def format_insights_as_natural_language(insights_data):
    """Formats the insights data as a natural language string."""
    output_string = ""

    if not insights_data or not isinstance(insights_data, list):
        return "No insights data available or data is not in the expected format."

    for insight in insights_data:
        for key, value in insight.items():
          if isinstance(value, list):
              for item in value:
                  if isinstance(item, dict) and 'action_type' in item and 'value' in item:
                        action_type = item['action_type']
                        action_value = item['value']
                        output_string += f"{action_type}: {action_value}\n"  # Correct formatting
                  elif isinstance(item, dict):
                        for sub_key, sub_value in item.items():
                            output_string += f"  {sub_key}: {sub_value}\n"
                  else:  # List of simple values
                        output_string += f"  {key}: {', '.join(map(str, value))}\n"  # Correct formatting for simple lists
                        break  
          else:
              output_string += f"{key}: {value}\n"
        output_string += "--------------------\n"

    return output_string

def get_max_campaign_insights_as_natural_language(campaign_id):
     try:
        my_account = AdAccount(AD_ACCOUNT_ID)

        campaign_insights = my_account.get_insights(
            fields=[
                'campaign_name',  #Campaign Name for identification
                'clicks', # Number of clicks on the ad
                'cpc',    # Cost per click
                'ctr',    # Click-through rate (important for ad effectiveness)
                'spend',   # Total amount spent on the campaign
                'reach', # Number of people reached
                'impressions',  # Number of times the ad was displayed
                'frequency',      # Average number of times each person saw the ad
                'conversions', # Number of conversions (if applicable to your objective)
                'cost_per_conversion', # Cost per conversion (if applicable)
                'conversion_values', #value of conversions
                'website_ctr', #website click through rate, if you have website conversions enabled
                'video_play_actions', #number of video plays, if using video ads
                'video_avg_time_watched_actions',  # Average video watch time (if using video ads)
                'objective', #the objective of the campaign
                'website_purchase_roas', # Return on ad spend for website purchases
                'cost_per_unique_click', # Cost per unique click (de-duplicates clicks)
                'actions', # Include actions (e.g., link clicks, video views) for more detailed analysis
                'action_values', #monetary value of actions
                'outbound_clicks', #all clicks leading out of facebook
                'cost_per_unique_outbound_click', #cost per click leading out of facebook
            ],
            params={
                "date_preset": "this_quarter",
                "breakdowns": ["age", "gender"],
                "level": "campaign",
                'filtering': [{'field': 'campaign.id', 'operator': 'EQUAL', 'value': campaign_id}],
                "limit": 1000
            }
        )
        campaign_insights_data = [insight._json for insight in campaign_insights]

        natural_language_string = format_insights_as_natural_language(campaign_insights_data)
        return natural_language_string

     except Exception as e:
        print(f"Error: {e}")
        return f"Error retrieving insights: {e}"

# Example usage:
natural_language_output = get_max_campaign_insights_as_natural_language("120213265876940368")
print(natural_language_output)

campaign_name: Talkative Campaign
clicks: 1
cpc: 10.218013
ctr: 0.406504
spend: 10.218013
reach: 246
impressions: 246
frequency: 1
video_view: 72
video_view: 3
objective: OUTCOME_SALES
cost_per_unique_click: 10.218013
post_engagement: 29
page_engagement: 29
video_view: 28
post_reaction: 1
date_start: 2024-10-01
date_stop: 2024-12-06
age: 18-24
gender: female
--------------------
campaign_name: Talkative Campaign
clicks: 208
cpc: 2.023895
ctr: 1.639861
spend: 420.970059
reach: 9441
impressions: 12684
frequency: 1.343502
link_click: 0.607064
video_view: 5043
video_view: 5
objective: OUTCOME_SALES
cost_per_unique_click: 2.26328
onsite_conversion.total_messaging_connection: 45
onsite_web_app_view_content: 36
onsite_conversion.messaging_first_reply: 39
landing_page_view: 5
onsite_web_view_content: 36
post_engagement: 2404
comment: 3
page_engagement: 2404
view_content: 36
onsite_conversion.messaging_user_depth_3_message_send: 20
post: 10
onsite_conversion.messaging_user_depth_2_message_send:

In [ ]:
def format_insights_as_natural_language(insights_data):
    """Formats the insights data as a natural language string."""
    output_string = ""

    if not insights_data or not isinstance(insights_data, list):
       return "No insights data available or data is not in the expected format."

    for insight in insights_data:
        for key, value in insight.items():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict): #handle list of dictionaries
                        for sub_key, sub_value in item.items():
                          output_string += f"{sub_key}: {sub_value}\n"
            else: #if its not a list
                output_string += f"{key}: {value}\n"    
        output_string += "--------------------\n"  # Separator between insights

    return output_string


def get_max_campaign_insights_as_natural_language(campaign_id):
     try:
        my_account = AdAccount(AD_ACCOUNT_ID)

        # Campaign level insights - Removed invalid fields
        campaign_insights = my_account.get_insights(
            fields=[
                'account_name', 'action_values', 'actions',  'ad_id', 'ad_name', 'adset_id',
                'adset_name',
                'auction_bid', 'auction_competitiveness',
                'auction_max_competitor_bid', 'buying_type', 'campaign_id', 'campaign_name',
                'canvas_avg_view_percent', 'canvas_avg_view_time', 'catalog_segment_actions',
                'catalog_segment_value', 'clicks', 'conversion_rate_ranking', 'conversion_values',
                'conversions', 
                 'cost_per_action_type', 'cost_per_conversion',
                'cost_per_estimated_ad_recallers', 'cost_per_inline_link_click',
                'cost_per_inline_post_engagement', 'cost_per_outbound_click',
                 'cost_per_unique_action_type',
                'cost_per_unique_click', 'cost_per_unique_conversion',
                'cost_per_unique_inline_link_click', 'cost_per_unique_outbound_click', 'cpc', 'cpp',
                'created_time', 'ctr',
                'engagement_rate_ranking', 'estimated_ad_recall_rate',
                'estimated_ad_recallers', 'frequency',
                'impressions', 'inline_link_click_ctr', 'inline_link_clicks', 'inline_post_engagement',
                'instant_experience_clicks_to_open', 'instant_experience_clicks_to_start',
                'instant_experience_outbound_clicks', 'interactive_component_tap',
                'mobile_app_purchase_roas', 'objective', 'optimization_goal', 'outbound_clicks',
                'outbound_clicks_ctr', 'place_page_name', 'purchase_roas',
                'reach', 'social_spend', 'spend',
                'updated_time', 'video_15_sec_watched_actions', 'video_30_sec_watched_actions',
                'video_avg_time_watched_actions', 'video_continuous_2_sec_watched_actions',
                'video_p100_watched_actions', 'video_p25_watched_actions', 'video_p50_watched_actions',
                'video_p75_watched_actions', 'video_p95_watched_actions', 'video_play_actions',
                'video_thruplay_watched_actions', 'video_time_watched_actions', 'website_ctr', 'website_purchase_roas'
            ],
            params={
                "date_preset": "this_quarter",
                "breakdowns": ["age", "gender"],
                "level": "campaign",
                'filtering': [{'field': 'campaign.id', 'operator': 'EQUAL', 'value': campaign_id}],
                "limit": 1000
            }
        )
        campaign_insights_data = [insight._json for insight in campaign_insights]

        natural_language_string = format_insights_as_natural_language(campaign_insights_data)
        return natural_language_string

     except Exception as e:
        print(f"Error: {e}")
        return f"Error retrieving insights: {e}"

# Example usage:
natural_language_output = get_max_campaign_insights_as_natural_language(campaign_id)
print(natural_language_output)

account_id: 2353530418276809
account_name: egenie haripriya
action_type: onsite_conversion.total_messaging_connection
value: 2
action_type: onsite_conversion.messaging_block
value: 1
action_type: onsite_conversion.messaging_first_reply
value: 2
action_type: post_engagement
value: 477
action_type: page_engagement
value: 477
action_type: onsite_conversion.messaging_user_depth_3_message_send
value: 1
action_type: onsite_conversion.messaging_conversation_started_7d
value: 2
action_type: video_view
value: 462
action_type: post_reaction
value: 10
action_type: link_click
value: 5
buying_type: AUCTION
campaign_id: 120212031328970368
campaign_name: Instagram post: Prices lower than any e-commerce...
clicks: 18
action_type: onsite_conversion.total_messaging_connection
value: 29.729824
action_type: onsite_conversion.messaging_block
value: 59.459649
action_type: video_view
value: 0.128701
action_type: link_click
value: 11.89193
action_type: page_engagement
value: 0.124653
action_type: onsite_conve

In [18]:
# Call the function
print_insights(insights_age)

In [11]:
insights_gender = my_account.get_insights(
    fields=[
        "campaign_name", "impressions", "clicks", "ctr", "cpc", "reach", 
        "frequency", "spend", "website_ctr", "video_avg_time_watched_actions", 
        "canvas_avg_view_time", "cost_per_unique_inline_link_click", 
        "cost_per_unique_outbound_click", "date_start", "date_stop", 
        "auction_competitiveness", "action_values"
        ],
    params={
            "date_preset": "this_quarter",
            "breakdowns": "gender",  
            "level": "campaign",
    }
)


In [12]:
print_insights(insights_gender)

Insight 1:
{
    "campaign_name": "Instagram post: Shipping at Price Lower Than...",
    "impressions": "2138",
    "clicks": "36",
    "ctr": "1.683817",
    "cpc": "2.896899",
    "reach": "1424",
    "frequency": "1.501404",
    "spend": "104.288377",
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.70159"
        }
    ],
    "cost_per_unique_inline_link_click": "7.44917",
    "date_start": "2024-10-01",
    "date_stop": "2024-12-02",
    "gender": "female"
}
--------------------------------------------------
Insight 2:
{
    "campaign_name": "Instagram post: Shipping at Price Lower Than...",
    "impressions": "62112",
    "clicks": "896",
    "ctr": "1.442555",
    "cpc": "3.033869",
    "reach": "47830",
    "frequency": "1.298599",
    "spend": "2718.346384",
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.722888"
        }
    ],
    "cost_per_unique_inline_link_click": "6.581953",
 

In [13]:
insights_country = my_account.get_insights(
    fields=[
        "campaign_name", "impressions", "clicks", "ctr", "cpc", "reach", 
        "frequency", "spend", "website_ctr", "video_avg_time_watched_actions", 
        "canvas_avg_view_time", "cost_per_unique_inline_link_click", 
        "cost_per_unique_outbound_click", "date_start", "date_stop", 
        "auction_competitiveness", "action_values"
        ],
    params={
            "date_preset": "this_quarter",
            "breakdowns": "country",  
            "level": "campaign",
    }
)

In [14]:
print_insights(insights_country)

Insight 1:
{
    "campaign_name": "Instagram post: Shipping at Price Lower Than...",
    "impressions": "64808",
    "clicks": "936",
    "ctr": "1.444266",
    "cpc": "3.040171",
    "reach": "49621",
    "frequency": "1.30606",
    "spend": "2845.6",
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.72059"
        }
    ],
    "cost_per_unique_inline_link_click": "6.617674",
    "date_start": "2024-10-01",
    "date_stop": "2024-12-02",
    "country": "IN"
}
--------------------------------------------------
Insight 2:
{
    "campaign_name": "Instagram post: Shipped to Karimnagar, Telangana...",
    "impressions": "3533",
    "clicks": "21",
    "ctr": "0.594396",
    "cpc": "7.101429",
    "reach": "3302",
    "frequency": "1.069958",
    "spend": "149.13",
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.169827"
        }
    ],
    "video_avg_time_watched_actions": [
        {
            

In [43]:
insights_device_platform = my_account.get_insights(
    fields=[
        "campaign_name", "impressions", "clicks", "ctr", "cpc", "reach", 
        "frequency", "spend", "website_ctr", "video_avg_time_watched_actions", 
        "canvas_avg_view_time", "cost_per_unique_inline_link_click", 
        "cost_per_unique_outbound_click", "date_start", "date_stop", 
        "auction_competitiveness"],
    params={
        "date_preset": "this_quarter",
        "breakdowns": "device_platform",
        "level": "campaign",
    }
)

# Print insights for review
for insight in insights_device_platform:
    print(insight)
    print("-" * 50)

<AdsInsights> {
    "campaign_name": "Instagram post: Shipping at Price Lower Than...",
    "clicks": "936",
    "cost_per_unique_inline_link_click": "6.617674",
    "cpc": "3.040171",
    "ctr": "1.444266",
    "date_start": "2024-10-01",
    "date_stop": "2024-12-01",
    "device_platform": "mobile_app",
    "frequency": "1.30606",
    "impressions": "64808",
    "reach": "49621",
    "spend": "2845.6",
    "website_ctr": [
        {
            "action_type": "link_click",
            "value": "0.72059"
        }
    ]
}
--------------------------------------------------
<AdsInsights> {
    "campaign_name": "Instagram post: Shipped to Karimnagar, Telangana...",
    "clicks": "21",
    "cost_per_unique_inline_link_click": "29.826",
    "cpc": "7.101429",
    "ctr": "0.594396",
    "date_start": "2024-10-01",
    "date_stop": "2024-12-01",
    "device_platform": "mobile_app",
    "frequency": "1.069958",
    "impressions": "3533",
    "reach": "3302",
    "spend": "149.13",
    "video

In [ ]:
def get_adsets_in_campaign(campaign_id):
    """
    Retrieves ad set metadata and a list of ad set IDs within a given campaign.
    """
    try:
        my_account = AdAccount(AD_ACCOUNT_ID)
        adsets_data = []  # List to store adset metadata
        adset_ids = []    # List to store just adset IDs

        adsets = my_account.get_insights(
            fields=[
                'adset_id',
                'adset_name',
                'optimization_goal'
            ],
            params={
                'level': 'adset',
                "date_preset": "this_quarter",
                'filtering': [{'field': 'campaign.id', 'operator': 'EQUAL', 'value': campaign_id}]
            }
        )

        for adset in adsets:
            adset_id = adset.get("adset_id")
            adsets_data.append({
                "id": adset_id,
                "name": adset.get("adset_name"),
                "effective_status": adset.get("effective_status"),
                # ... other fields
            })
            adset_ids.append(adset_id)  # Add adset ID to the list


        return adsets_data, adset_ids #return both meta data and list of ids

    except Exception as e:
        print(f"Error retrieving ad sets: {e}")
        return None, None  # Return None for both in case of error


# Example usage:
campaign_id = "120213265876940368"
adsets_metadata, adset_ids_list = get_adsets_in_campaign(campaign_id)


if adsets_metadata and adset_ids_list:
  print("Adset Metadata:")
  for adset_data in adsets_metadata:
      print(adset_data)
      print("-" * 20)

  print("\nList of Adset IDs:")
  print(adset_ids_list)

else:
    print(f"No ad sets found for campaign ID: {campaign_id}")

Adset Metadata:
{'id': '120213297774640368', 'name': 'Talkative ad set Messages', 'effective_status': None}
--------------------

List of Adset IDs:
['120213297774640368']


In [ ]:
def get_adset_insights(adset_ids):
    """Fetches insights for multiple ad sets and returns them as a dictionary."""

    try:
        my_account = AdAccount(AD_ACCOUNT_ID)
        adset_insights = {}

        for adset_id in adset_ids:
            insights = my_account.get_insights(
                fields=[
                    'adset_name',
                    'clicks',
                    'cpc',
                    'ctr',
                    'spend',
                    'reach',
                    'impressions',
                    'frequency',
                    'conversions',
                    'cost_per_conversion',
                    'conversion_values',  # Include conversion values
                    'website_ctr',
                    'video_play_actions',  # Video play actions
                   'video_avg_time_watched_actions', #average watch time
                    'objective',
                    'website_purchase_roas',
                    'cost_per_unique_click',
                    'outbound_clicks',
                     'cost_per_unique_outbound_click',
                    'actions',
                     'action_values', # Monetary value of actions
                ],
                params={
                    'level': 'adset',
                     "date_preset": "this_quarter",#can be lifetime or last_month etc
                    'filtering': [{'field': 'adset.id', 'operator': 'EQUAL', 'value': adset_id}],
                    "limit": 100  # Adjust limit as needed
                }
            )
              # Convert insights data to JSON for easier handling
            adset_insights[adset_id] = [insight._json for insight in insights]


        return adset_insights

    except Exception as e:
        print(f"Error: {e}")
        return f"Error retrieving insights: {e}"

def format_adset_insights(insights_data):
    """Formats adset insights data into a readable string."""

    output = ""
    for adset_id, insights in insights_data.items():
        output += f"Adset ID: {adset_id}\n"
        for insight in insights:
            for key, value in insight.items():
                output += f"  {key}: {value}\n"  
        output += "--------------------\n"
    return output

def get_adsets_in_campaign(campaign_id):
    """
    Retrieves ad set metadata and a list of ad set IDs within a given campaign.
    """
    try:
        my_account = AdAccount(AD_ACCOUNT_ID)
        adsets_data = []  # List to store adset metadata
        adset_ids = []    # List to store just adset IDs

        adsets = my_account.get_insights(
            fields=[
                'adset_id',
                'adset_name',
                'optimization_goal'
            ],
            params={
                'level': 'adset',
                "date_preset": "this_quarter",
                'filtering': [{'field': 'campaign.id', 'operator': 'EQUAL', 'value': campaign_id}]
            }
        )

        for adset in adsets:
            adset_id = adset.get("adset_id")
            adsets_data.append({
                "id": adset_id,
                "name": adset.get("adset_name"),
                "effective_status": adset.get("effective_status"),
                # ... other fields
            })
            adset_ids.append(adset_id)  # Add adset ID to the list


        return adsets_data, adset_ids #return both meta data and list of ids

    except Exception as e:
        print(f"Error retrieving ad sets: {e}")
        return None, None  # Return None for both in case of error


campaign_id = "120213265876940368"
_, adset_ids = get_adsets_in_campaign(campaign_id)

if not adset_ids:
    print(f"No ad sets found for campaign ID: {campaign_id}")

insights_data = get_adset_insights(adset_ids)
formatted_insights = format_adset_insights(insights_data)
print(formatted_insights)

Adset ID: 120213297774640368
  adset_name: Talkative ad set Messages
  clicks: 184
  cpc: 2.496413
  ctr: 1.534484
  spend: 459.34
  reach: 9522
  impressions: 11991
  frequency: 1.259294
  website_ctr: [{'action_type': 'link_click', 'value': '0.533734'}]
  video_play_actions: [{'action_type': 'video_view', 'value': '967'}]
  video_avg_time_watched_actions: [{'action_type': 'video_view', 'value': '2'}]
  objective: OUTCOME_SALES
  cost_per_unique_click: 2.734167
  actions: [{'action_type': 'onsite_conversion.total_messaging_connection', 'value': '36'}, {'action_type': 'onsite_web_app_view_content', 'value': '21'}, {'action_type': 'onsite_conversion.messaging_first_reply', 'value': '33'}, {'action_type': 'landing_page_view', 'value': '3'}, {'action_type': 'onsite_web_view_content', 'value': '21'}, {'action_type': 'post_engagement', 'value': '279'}, {'action_type': 'comment', 'value': '3'}, {'action_type': 'page_engagement', 'value': '279'}, {'action_type': 'view_content', 'value': '21'}